# EfficientNet-B3 Fine-tuning - HAM10000 (Colab GPU)

Calisma sirasi:
1. Runtime > Change runtime type > GPU sec.
2. Asagidaki hucreleri sirayla calistir.
3. Istendiginde `kaggle.json` ve `train.csv`, `val.csv`, `test.csv` dosyalarini yukle.

In [ ]:
import torch
print('GPU var mi:', torch.cuda.is_available())
print('Cihaz:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Kaggle API ile veri setini indir
Asagidaki hucre calisinca `kaggle.json` dosyani yuklemeni isteyecek (Kaggle hesabinda Account > Create New API Token ile indirdigin dosya).

In [ ]:
from google.colab import files
import os

os.makedirs('/root/.kaggle', exist_ok=True)
uploaded = files.upload()  # kaggle.json sec
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data
!unzip -q -o /content/data/skin-cancer-mnist-ham10000.zip -d /content/data
print('Indirme tamamlandi.')

## 2. Train/val/test bolme dosyalarini yukle
Yerel PC'de `src/split_data.py` ile ürettigimiz `train.csv`, `val.csv`, `test.csv` dosyalarini (data/ klasorunden) burada yukle.

In [ ]:
from google.colab import files
uploaded = files.upload()  # train.csv, val.csv, test.csv sec (ucunu birden)
import shutil
for name in uploaded:
    shutil.move(name, f'/content/data/{name}')
print('Split dosyalari yuklendi.')

## 3. Kod (dataset, transforms, model, train, grad_cam)
Yerel PC'deki `src/` klasorundeki kodlarin birebir ayni; Colab'de tek dosyada tekrar tanimliyoruz.

In [ ]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, image_dirs, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dirs = image_dirs
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _find_image_path(self, image_id):
        filename = f"{image_id}.jpg"
        for directory in self.image_dirs:
            candidate = os.path.join(directory, filename)
            if os.path.exists(candidate):
                return candidate
        raise FileNotFoundError(f"{filename} not found in {self.image_dirs}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self._find_image_path(row["image_id"])
        image = Image.open(image_path).convert("RGB")
        label = CLASS_TO_IDX[row["dx"]]
        if self.transform:
            image = self.transform(image)
        return image, label

def load_metadata(metadata_csv):
    return pd.read_csv(metadata_csv)

In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 300

def get_train_transforms():
    return transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def get_eval_transforms():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

def build_model(num_classes, freeze_backbone=True):
    weights = EfficientNet_B3_Weights.IMAGENET1K_V1
    model = efficientnet_b3(weights=weights)
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

In [ ]:
import copy
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = None
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False

def run_epoch(model, loader, criterion, optimizer, device, train):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    torch.set_grad_enabled(train)
    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        if train:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

## 4. Egitimi baslat

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

image_dirs = ['/content/data/HAM10000_images_part_1', '/content/data/HAM10000_images_part_2']
train_df = load_metadata('/content/data/train.csv')
val_df = load_metadata('/content/data/val.csv')

train_ds = HAM10000Dataset(train_df, image_dirs, transform=get_train_transforms())
val_ds = HAM10000Dataset(val_df, image_dirs, transform=get_eval_transforms())

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

model = build_model(num_classes=len(CLASS_NAMES), freeze_backbone=True).to(device)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CLASS_NAMES)),
    y=train_df['dx'].map(CLASS_TO_IDX),
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

early_stopping = EarlyStopping(patience=5)
best_state = None
epochs = 30

for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
    print(f'Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}')

    improved = early_stopping.step(val_loss)
    if improved:
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, '/content/best_model.pth')
        print('  -> val_loss improved, model saved.')

    if early_stopping.should_stop:
        print(f'Early stopping triggered at epoch {epoch}.')
        break

if best_state is not None:
    model.load_state_dict(best_state)

## 5. En iyi modeli indir
Bu dosyayi (`best_model.pth`) indirip yerel PC'deki `outputs/` klasorune koyacaksin, Grad-CAM icin kullanacagiz.

In [ ]:
from google.colab import files
files.download('/content/best_model.pth')